# 07 — Retrieval Evaluation

This notebook implements the quantitative evaluation stage of the **CLARA** retrieval pipeline.

The previous notebook demonstrated semantic search qualitatively by inspecting the legal chunks returned for individual questions. This notebook extends that validation by evaluating retrieval performance against a predefined set of questions and expected legal sources.

The objective is to determine whether the vector retrieval component places the relevant legal article among the highest-ranked results.

The evaluation artefacts are stored under:

```text
data/evaluation/
```

Typical outputs include question definitions, detailed retrieval results, aggregate metrics, and a summary of the evaluation run.

## 1. Imports

This block loads the libraries required for JSON and CSV processing, filesystem operations, HTTP communication, numerical calculations, timing, and tabular analysis of the retrieval results.

In [1]:
from pathlib import Path
import json
import time

import pandas as pd
import requests


## 2. Evaluation Configuration

The project paths and retrieval parameters used during evaluation are defined here.

The evaluation must use the same Qdrant collection, embedding model, and retrieval configuration as the operational CLARA pipeline so that the measured results reflect the actual search behaviour of the system.

In [2]:
PROJECT_ROOT = Path("..").resolve()

EVALUATION_DIR = PROJECT_ROOT / "data" / "evaluation"
EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "bge-m3"

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "clara_bge_m3"

TOP_K_VALUES = [1, 3, 5, 10]

print(f"Evaluation: {EVALUATION_DIR}")
print(f"Ollama:     {OLLAMA_URL}")
print(f"Qdrant:     {QDRANT_URL}")
print(f"Collection: {COLLECTION_NAME}")


Evaluation: C:\Users\user\Documents\GitHub\clara-rag-pt\data\evaluation
Ollama:     http://localhost:11434
Qdrant:     http://localhost:6333
Collection: clara_bge_m3


## Ground truth

O ficheiro `evaluation_questions.json` contém as perguntas de avaliação.

Cada pergunta pode definir:
- `relevant_documents`: diplomas esperados;
- `relevant_articles`: pares documento/artigo esperados.

Exemplo:

```json
{
  "id": "Q001",
  "question": "Como são tributados os rendimentos prediais?",
  "relevant_documents": ["CIRS"],
  "relevant_articles": [
    {"document_id": "CIRS", "article": "Artigo 72.º"}
  ]
}
```

A relevância é considerada satisfeita se o resultado corresponder a um artigo explicitamente esperado. Caso não sejam fornecidos artigos, é usada a relevância ao nível do documento.


## 2. Evaluation Configuration

The project paths and retrieval parameters used during evaluation are defined here.

The evaluation must use the same Qdrant collection, embedding model, and retrieval configuration as the operational CLARA pipeline so that the measured results reflect the actual search behaviour of the system.

In [3]:
EVALUATION_FILE = (
    EVALUATION_DIR
    / "evaluation_questions.json"
)

if not EVALUATION_FILE.exists():

    template = [
        {
            "id": "Q001",
            "question": "Como são tributados os rendimentos prediais?",
            "relevant_documents": ["CIRS"],
            "relevant_articles": []
        },
        {
            "id": "Q002",
            "question": "Como é determinado o valor patrimonial tributário dos imóveis?",
            "relevant_documents": ["CIMI"],
            "relevant_articles": []
        },
        {
            "id": "Q003",
            "question": "Quais são as principais obrigações declarativas em IVA?",
            "relevant_documents": ["CIVA"],
            "relevant_articles": []
        },
        {
            "id": "Q004",
            "question": "Que benefícios fiscais estão previstos para imóveis?",
            "relevant_documents": ["EBF"],
            "relevant_articles": []
        },
        {
            "id": "Q005",
            "question": "Quais são as regras relativas às infrações tributárias?",
            "relevant_documents": ["RGIT"],
            "relevant_articles": []
        }
    ]

    with open(
        EVALUATION_FILE,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            template,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(
        "Foi criado um ficheiro de exemplo:"
    )
    print(EVALUATION_FILE)
    print(
        "\nRevê o ground truth antes de executar "
        "a avaliação definitiva."
    )

else:
    print(
        f"Ficheiro encontrado: {EVALUATION_FILE}"
    )


Foi criado um ficheiro de exemplo:
C:\Users\user\Documents\GitHub\clara-rag-pt\data\evaluation\evaluation_questions.json

Revê o ground truth antes de executar a avaliação definitiva.


## 3. Evaluation Question Set

The retrieval benchmark is based on a predefined collection of natural-language questions.

Each question is associated with an expected legal source, allowing the ranking produced by semantic search to be compared against a known relevant article or document.

This creates a small labelled retrieval test set for CLARA.

In [4]:
with open(
    EVALUATION_FILE,
    "r",
    encoding="utf-8"
) as f:
    evaluation_questions = json.load(f)

print(
    f"Perguntas de avaliação: "
    f"{len(evaluation_questions)}"
)

pd.DataFrame(evaluation_questions)


Perguntas de avaliação: 5


,id,question,relevant_documents,relevant_articles
0,Q001,Como são tributados os rendimentos prediais?,[CIRS],[]
1,Q002,Como é determinado o valor patrimonial tributá...,[CIMI],[]
2,Q003,Quais são as principais obrigações declarativa...,[CIVA],[]
3,Q004,Que benefícios fiscais estão previstos para im...,[EBF],[]
4,Q005,Quais são as regras relativas às infrações tri...,[RGIT],[]


## Funções de recuperação


## 4. Query Embedding Function

Each evaluation question is converted into a semantic vector using **BGE-M3** through Ollama.

The same embedding model used to index the legal corpus is used here to ensure that evaluation queries and stored document vectors occupy the same semantic space.

In [5]:
def generate_embedding(
    text: str,
    model: str = EMBEDDING_MODEL
) -> list[float]:

    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={
            "model": model,
            "input": text
        },
        timeout=120
    )

    response.raise_for_status()

    data = response.json()

    embeddings = data.get("embeddings")

    if not embeddings:
        raise ValueError(
            "Ollama não devolveu embeddings."
        )

    return embeddings[0]


## 2. Evaluation Configuration

The project paths and retrieval parameters used during evaluation are defined here.

The evaluation must use the same Qdrant collection, embedding model, and retrieval configuration as the operational CLARA pipeline so that the measured results reflect the actual search behaviour of the system.

In [6]:
def vector_search(
    question: str,
    top_k: int = 10
) -> list[dict]:

    query_vector = generate_embedding(
        question
    )

    response = requests.post(
        (
            f"{QDRANT_URL}/collections/"
            f"{COLLECTION_NAME}/points/search"
        ),
        json={
            "vector": query_vector,
            "limit": top_k,
            "with_payload": True,
            "with_vector": False
        },
        timeout=120
    )

    response.raise_for_status()

    return response.json()["result"]


## Definição de relevância

Quando `relevant_articles` contém valores, a avaliação é feita ao nível do artigo.

Quando `relevant_articles` está vazio, a avaliação é feita ao nível do diploma.


### Evaluation Step 7

This block performs an additional operation in the retrieval evaluation workflow.

It contributes to measuring, validating, or persisting the behaviour of the CLARA semantic retriever against the labelled legal-question dataset.

In [7]:
def expected_relevant_keys(
    evaluation_item: dict
) -> set[tuple]:

    relevant_articles = evaluation_item.get(
        "relevant_articles",
        []
    )

    if relevant_articles:

        return {
            (
                item["document_id"],
                item["article"]
            )
            for item in relevant_articles
        }

    return {
        (document_id, None)
        for document_id in evaluation_item.get(
            "relevant_documents",
            []
        )
    }


def result_key(
    result: dict,
    article_level: bool
) -> tuple:

    payload = result["payload"]

    if article_level:
        return (
            payload["document_id"],
            payload["article"]
        )

    return (
        payload["document_id"],
        None
    )


### Evaluation Step 8

This block performs an additional operation in the retrieval evaluation workflow.

It contributes to measuring, validating, or persisting the behaviour of the CLARA semantic retriever against the labelled legal-question dataset.

In [8]:
def is_relevant(
    result: dict,
    evaluation_item: dict
) -> bool:

    relevant_articles = evaluation_item.get(
        "relevant_articles",
        []
    )

    article_level = bool(
        relevant_articles
    )

    expected = expected_relevant_keys(
        evaluation_item
    )

    return (
        result_key(
            result,
            article_level=article_level
        )
        in expected
    )


## Métricas


## 6. Hit@K

**Hit@K** measures whether at least one relevant result appears within the first `K` retrieved items.

For a question \(q\):

```text
Hit@K(q) = 1   if a relevant item appears in the Top-K
           0   otherwise
```

The aggregate score is the mean across all evaluation questions.

This metric answers a practical RAG question: *Does the retriever provide the generator with the expected legal source within the available context window?*

In [9]:
def hit_at_k(
    results: list[dict],
    evaluation_item: dict,
    k: int
) -> int:

    return int(
        any(
            is_relevant(
                result,
                evaluation_item
            )
            for result in results[:k]
        )
    )


def recall_at_k(
    results: list[dict],
    evaluation_item: dict,
    k: int
) -> float:

    relevant_articles = evaluation_item.get(
        "relevant_articles",
        []
    )

    article_level = bool(
        relevant_articles
    )

    expected = expected_relevant_keys(
        evaluation_item
    )

    if not expected:
        return 0.0

    retrieved_relevant = {
        result_key(
            result,
            article_level=article_level
        )
        for result in results[:k]
        if is_relevant(
            result,
            evaluation_item
        )
    }

    return (
        len(retrieved_relevant)
        /
        len(expected)
    )


def reciprocal_rank(
    results: list[dict],
    evaluation_item: dict
) -> float:

    for rank, result in enumerate(
        results,
        start=1
    ):

        if is_relevant(
            result,
            evaluation_item
        ):
            return 1.0 / rank

    return 0.0


## Teste das métricas numa pergunta


## 3. Evaluation Question Set

The retrieval benchmark is based on a predefined collection of natural-language questions.

Each question is associated with an expected legal source, allowing the ranking produced by semantic search to be compared against a known relevant article or document.

This creates a small labelled retrieval test set for CLARA.

In [10]:
test_item = evaluation_questions[0]

test_results = vector_search(
    test_item["question"],
    top_k=max(TOP_K_VALUES)
)

print(
    f"Pergunta: {test_item['question']}"
)

print(
    "Relevantes:",
    expected_relevant_keys(test_item)
)

for rank, result in enumerate(
    test_results,
    start=1
):

    payload = result["payload"]

    print(
        f"{rank:2} | "
        f"{payload['document_id']:8} | "
        f"{payload['article']:15} | "
        f"{result['score']:.4f} | "
        f"{'RELEVANTE' if is_relevant(result, test_item) else ''}"
    )


Pergunta: Como são tributados os rendimentos prediais?
Relevantes: {('CIRS', None)}
 1 | CIRS     | Artigo 8.º      | 0.7041 | RELEVANTE
 2 | EBF      | Artigo 45.º-C   | 0.6563 | 
 3 | CIRS     | Artigo 16.º     | 0.6515 | RELEVANTE
 4 | EBF      | Artigo 74.º-A   | 0.6382 | 
 5 | CIRS     | Artigo 72.º     | 0.6319 | RELEVANTE
 6 | CIRS     | Artigo 72.º     | 0.6298 | RELEVANTE
 7 | CIRC     | Artigo 56.º     | 0.6227 | 
 8 | CIMI     | Artigo 12.º     | 0.6009 | 
 9 | CIRS     | Artigo 41.º     | 0.5988 | RELEVANTE
10 | EBF      | Artigo 59.º-G   | 0.5934 | 


## Executar avaliação completa


## 3. Evaluation Question Set

The retrieval benchmark is based on a predefined collection of natural-language questions.

Each question is associated with an expected legal source, allowing the ranking produced by semantic search to be compared against a known relevant article or document.

This creates a small labelled retrieval test set for CLARA.

In [11]:
evaluation_rows = []
retrieval_rows = []

MAX_K = max(TOP_K_VALUES)

for index, item in enumerate(
    evaluation_questions,
    start=1
):

    question_id = item["id"]
    question = item["question"]

    print(
        f"{index}/{len(evaluation_questions)} "
        f"— {question_id}"
    )

    start_time = time.perf_counter()

    results = vector_search(
        question,
        top_k=MAX_K
    )

    latency_ms = (
        time.perf_counter()
        -
        start_time
    ) * 1000

    row = {
        "id": question_id,
        "question": question,
        "latency_ms": latency_ms,
        "mrr": reciprocal_rank(
            results,
            item
        )
    }

    for k in TOP_K_VALUES:

        row[f"hit@{k}"] = hit_at_k(
            results,
            item,
            k
        )

        row[f"recall@{k}"] = recall_at_k(
            results,
            item,
            k
        )

    evaluation_rows.append(row)

    for rank, result in enumerate(
        results,
        start=1
    ):

        payload = result["payload"]

        retrieval_rows.append({
            "question_id": question_id,
            "question": question,
            "rank": rank,
            "score": result["score"],
            "document_id": payload[
                "document_id"
            ],
            "article": payload[
                "article"
            ],
            "article_title": payload.get(
                "article_title"
            ),
            "page_start": payload[
                "page_start"
            ],
            "page_end": payload[
                "page_end"
            ],
            "chunk_id": payload[
                "chunk_id"
            ],
            "relevant": is_relevant(
                result,
                item
            )
        })


1/5 — Q001
2/5 — Q002
3/5 — Q003
4/5 — Q004
5/5 — Q005


## 11. Detailed Retrieval Results

Question-level retrieval results are converted into a Pandas DataFrame for analysis.

This representation makes it easier to inspect expected sources, retrieved articles, ranking positions, similarity scores, and whether each question was successfully resolved within the selected Top-K.

In [12]:
df_evaluation = pd.DataFrame(
    evaluation_rows
)

df_evaluation


,id,question,latency_ms,mrr,hit@1,recall@1,hit@3,recall@3,hit@5,recall@5,hit@10,recall@10
0,Q001,Como são tributados os rendimentos prediais?,464.2494,1.0,1,1.0,1,1.0,1,1.0,1,1.0
1,Q002,Como é determinado o valor patrimonial tributá...,473.4475,0.5,0,0.0,1,1.0,1,1.0,1,1.0
2,Q003,Quais são as principais obrigações declarativa...,491.5522,1.0,1,1.0,1,1.0,1,1.0,1,1.0
3,Q004,Que benefícios fiscais estão previstos para im...,449.1998,1.0,1,1.0,1,1.0,1,1.0,1,1.0
4,Q005,Quais são as regras relativas às infrações tri...,381.8280,1.0,1,1.0,1,1.0,1,1.0,1,1.0


## Métricas agregadas


## 6. Hit@K

**Hit@K** measures whether at least one relevant result appears within the first `K` retrieved items.

For a question \(q\):

```text
Hit@K(q) = 1   if a relevant item appears in the Top-K
           0   otherwise
```

The aggregate score is the mean across all evaluation questions.

This metric answers a practical RAG question: *Does the retriever provide the generator with the expected legal source within the available context window?*

In [13]:
summary = {
    "questions": len(df_evaluation),
    "MRR": df_evaluation["mrr"].mean(),
    "mean_latency_ms": (
        df_evaluation["latency_ms"].mean()
    )
}

for k in TOP_K_VALUES:

    summary[f"Hit@{k}"] = (
        df_evaluation[
            f"hit@{k}"
        ].mean()
    )

    summary[f"Recall@{k}"] = (
        df_evaluation[
            f"recall@{k}"
        ].mean()
    )

df_summary = pd.DataFrame(
    [summary]
)

df_summary


,questions,MRR,mean_latency_ms,Hit@1,Recall@1,Hit@3,Recall@3,Hit@5,Recall@5,Hit@10,Recall@10
0,5,0.9,452.05538,0.8,0.8,1.0,1.0,1.0,1.0,1.0,1.0


## 6. Hit@K

**Hit@K** measures whether at least one relevant result appears within the first `K` retrieved items.

For a question \(q\):

```text
Hit@K(q) = 1   if a relevant item appears in the Top-K
           0   otherwise
```

The aggregate score is the mean across all evaluation questions.

This metric answers a practical RAG question: *Does the retriever provide the generator with the expected legal source within the available context window?*

In [14]:
print(
    f"Perguntas: {summary['questions']}"
)

print(
    f"MRR: {summary['MRR']:.4f}"
)

for k in TOP_K_VALUES:

    print(
        f"Hit@{k}: "
        f"{summary[f'Hit@{k}']:.4f}"
    )

    print(
        f"Recall@{k}: "
        f"{summary[f'Recall@{k}']:.4f}"
    )

print(
    f"Latência média: "
    f"{summary['mean_latency_ms']:.2f} ms"
)


Perguntas: 5
MRR: 0.9000
Hit@1: 0.8000
Recall@1: 0.8000
Hit@3: 1.0000
Recall@3: 1.0000
Hit@5: 1.0000
Recall@5: 1.0000
Hit@10: 1.0000
Recall@10: 1.0000
Latência média: 452.06 ms


## Inspeção dos resultados por pergunta


## 11. Detailed Retrieval Results

Question-level retrieval results are converted into a Pandas DataFrame for analysis.

This representation makes it easier to inspect expected sources, retrieved articles, ranking positions, similarity scores, and whether each question was successfully resolved within the selected Top-K.

In [15]:
df_retrieval = pd.DataFrame(
    retrieval_rows
)

df_retrieval[
    [
        "question_id",
        "rank",
        "score",
        "document_id",
        "article",
        "article_title",
        "relevant"
    ]
].head(50)


,question_id,rank,score,document_id,article,article_title,relevant
0,Q001,1,0.704147,CIRS,Artigo 8.º,Rendimentos da categoria F,True
1,Q001,2,0.656335,EBF,Artigo 45.º-C,"(Aditado pelo Decreto-Lei n.º 97/2026, de 20 d...",False
2,Q001,3,0.651471,CIRS,Artigo 16.º,As modificações do Código,True
3,Q001,4,0.638217,EBF,Artigo 74.º-A,Transferência de imóveis de alojamento local p...,False
4,Q001,5,0.631915,CIRS,Artigo 72.º,Taxas especiais,True
5,Q001,6,0.629810,CIRS,Artigo 72.º,Taxas especiais,True
6,Q001,7,0.622691,CIRC,Artigo 56.º,Rendimentos não imputáveis a estabelecimento e...,False
7,Q001,8,0.600886,CIMI,Artigo 12.º,Conceito de matrizes prediais,False
8,Q001,9,0.598820,CIRS,Artigo 41.º,Deduções,True
9,Q001,10,0.593377,EBF,Artigo 59.º-G,Entidades de gestão florestal e unidades de ge...,False


## Perguntas em que o resultado relevante não aparece no Top-5


## 6. Hit@K

**Hit@K** measures whether at least one relevant result appears within the first `K` retrieved items.

For a question \(q\):

```text
Hit@K(q) = 1   if a relevant item appears in the Top-K
           0   otherwise
```

The aggregate score is the mean across all evaluation questions.

This metric answers a practical RAG question: *Does the retriever provide the generator with the expected legal source within the available context window?*

In [16]:
failed_top5 = df_evaluation[
    df_evaluation["hit@5"] == 0
][
    [
        "id",
        "question",
        "mrr",
        "latency_ms"
    ]
]

failed_top5


,id,question,mrr,latency_ms


## Primeiro resultado relevante por pergunta


## 11. Detailed Retrieval Results

Question-level retrieval results are converted into a Pandas DataFrame for analysis.

This representation makes it easier to inspect expected sources, retrieved articles, ranking positions, similarity scores, and whether each question was successfully resolved within the selected Top-K.

In [17]:
first_relevant_rows = []

for question_id in df_retrieval[
    "question_id"
].unique():

    subset = df_retrieval[
        (
            df_retrieval["question_id"]
            ==
            question_id
        )
        &
        (
            df_retrieval["relevant"]
        )
    ]

    if len(subset):

        first = subset.sort_values(
            "rank"
        ).iloc[0]

        first_relevant_rows.append({
            "question_id": question_id,
            "rank": int(first["rank"]),
            "document_id": first[
                "document_id"
            ],
            "article": first[
                "article"
            ],
            "score": first[
                "score"
            ]
        })

    else:

        first_relevant_rows.append({
            "question_id": question_id,
            "rank": None,
            "document_id": None,
            "article": None,
            "score": None
        })

df_first_relevant = pd.DataFrame(
    first_relevant_rows
)

df_first_relevant


,question_id,rank,document_id,article,score
0,Q001,1,CIRS,Artigo 8.º,0.704147
1,Q002,2,CIMI,Artigo 38.º,0.714575
2,Q003,1,CIVA,Artigo 29.º,0.640659
3,Q004,1,EBF,Artigo 44.º-B,0.660340
4,Q005,1,RGIT,Artigo 2.º,0.659754


## Guardar resultados

São guardados três ficheiros:
- `evaluation_metrics.csv`
- `retrieval_results.csv`
- `evaluation_summary.json`


## 2. Evaluation Configuration

The project paths and retrieval parameters used during evaluation are defined here.

The evaluation must use the same Qdrant collection, embedding model, and retrieval configuration as the operational CLARA pipeline so that the measured results reflect the actual search behaviour of the system.

In [18]:
METRICS_FILE = (
    EVALUATION_DIR
    / "evaluation_metrics.csv"
)

RETRIEVAL_FILE = (
    EVALUATION_DIR
    / "retrieval_results.csv"
)

SUMMARY_FILE = (
    EVALUATION_DIR
    / "evaluation_summary.json"
)

df_evaluation.to_csv(
    METRICS_FILE,
    index=False,
    encoding="utf-8-sig"
)

df_retrieval.to_csv(
    RETRIEVAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )

print(METRICS_FILE)
print(RETRIEVAL_FILE)
print(SUMMARY_FILE)


C:\Users\user\Documents\GitHub\clara-rag-pt\data\evaluation\evaluation_metrics.csv
C:\Users\user\Documents\GitHub\clara-rag-pt\data\evaluation\retrieval_results.csv
C:\Users\user\Documents\GitHub\clara-rag-pt\data\evaluation\evaluation_summary.json


## Validação final

Os resultados devem ser interpretados apenas depois de o `evaluation_questions.json` conter um ground truth revisto manualmente.

Com poucas perguntas, as métricas são úteis como teste funcional, mas não constituem ainda uma avaliação robusta do sistema.


## Interpreting the Evaluation

Retrieval evaluation isolates the search component from answer generation.

A strong retrieval result does not guarantee that the language model will always produce a correct legal answer, but poor retrieval places an upper bound on the quality of the final RAG system: if the relevant legal provision is not retrieved, the generator cannot reliably ground its response in that provision.

For this reason, retrieval metrics should be analysed separately from generation quality.

## Next Step

The next notebook is:

```text
08_rag_generation_amalia.ipynb
```

It combines the evaluated retrieval pipeline with **AMALIA-9B** to implement the complete CLARA RAG workflow:

```text
Question
   ↓
BGE-M3
   ↓
Qdrant Retrieval
   ↓
Legal Context
   ↓
AMALIA-9B
   ↓
Grounded Answer
```

This final notebook validates the end-to-end transition from natural-language legal questions to answers generated from the retrieved legislation.
